# Split&Splat — Stage 1, bước 2 → 4: lan truyền mask để có ID nhất quán từng vật

Notebook này chạy **đúng các lệnh trong README của Split&Splat** (https://github.com/LTTM/Split_and_Splat) sau khi đã có mask SAM 2 của bước 1:

| README (Stage 1) | Lệnh gốc | Ô trong notebook |
|---|---|---|
| bước 1 | `python ./sam2/auto_seg.py` | **đã làm ở notebook trước** (mask 2D từng frame, 4 cấu hình SAM 2) |
| **bước 2** | `python ./sam2/mask_propagation.py -dataset <scene>` *(bản dành cho COLMAP/LERF; ScanNet mới dùng `mask_propagation_scanet.py`)* | mục 6 |
| **bước 3** | `mv ./output/<scene>_masks ./data/<scene>/masks` | mục 7 |
| **bước 4** | `./bash_dir_utils/prepare_folder.sh <scene>` | mục 8 |

```
mask SAM 2 (notebook trước) ─┐
COLMAP + depth Murre + point cloud 3DGS (notebook chuẩn bị) ─┴─► bước 2: gán ID nhất quán ─► bước 3 ─► bước 4: thư mục từng vật (masks/<id>/…)
```

**Đầu vào cần có cho mỗi scene** (README + code `mask_propagation.py`):
`data/<scene>/images/<tên>.JPEG`, `data/<scene>/sparse/0/` (COLMAP, camera PINHOLE, kèm `points3D.ply`), `data/<scene>/depth/<tên>_pred.npy`,
và `output/<scene>_autoseg_mask/<tên>/<k>.png` (mask bước 1). Notebook chuẩn bị tạo ra 3 thứ đầu; mask bước 1 lấy từ 2 file zip trên Drive.

> ⚠️ Mã nguồn của tác giả được giữ **nguyên văn** (clone đúng commit, không sửa file nào). Notebook chỉ dựng môi trường (Python 3.12, torch 2.5.1 + CUDA 12.4 như `environment.yml`), sắp xếp thư mục đầu vào, rồi gọi đúng các lệnh trên.

## 0. Kiểm tra GPU

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "Chưa có GPU: Runtime ▸ Change runtime type ▸ chọn GPU (khuyên dùng L4) rồi chạy lại."
print("GPU:", gpu)

## 1. Tham số *(sửa ở đây)*

In [ ]:
#@title 1. Tham số { display-mode: "form" }

#@markdown ### Mask từ Stage 1 bước 1 (zip do notebook SAM 2 tạo ra)
STAGE1_GDRIVE_IDS = "1A98WGR3DArEXYEXTWu4tQGHnKq7fn_5G, 1SJCD8mHqvoxgssohEYwtglalsY9y84oH"  #@param {type:"string"}
STAGE1_ZIP_DIR    = "/content/stage1_zips"  #@param {type:"string"}
#@markdown Mặc định là 2 zip `figurines_sam2_masks.zip` và `teatime_sam2_masks.zip` trong Drive của bạn. Cũng có thể tự upload zip vào `STAGE1_ZIP_DIR`.

#@markdown ### Đầu vào từ notebook chuẩn bị (COLMAP + depth Murre + point cloud 3DGS)
PREP_GDRIVE_IDS = ""  #@param {type:"string"}
PREP_ZIP_DIR    = "/content/prep_zips"  #@param {type:"string"}
#@markdown Điền ID các file `<scene>_prep.zip` (kết quả notebook chuẩn bị) hoặc upload thẳng vào `PREP_ZIP_DIR`.

#@markdown ### Chọn scene
SCENES = "all"  #@param {type:"string"}
#@markdown `all` hoặc liệt kê tên (vd `teatime`). Chỉ chạy scene có đủ cả mask Stage 1 lẫn đầu vào chuẩn bị.

#@markdown ### Mã nguồn tác giả
REPO_URL = "https://github.com/LTTM/Split_and_Splat.git"  #@param {type:"string"}
REPO_REF = "9b7ba906511f75b87c438b4cf30ae3d1d5302f61"  #@param {type:"string"}
SAM2_GIT_REF = "2b90b9f5ceec907a1c18123530e92e794ad901a4"  #@param {type:"string"}
#@markdown `REPO_REF` là commit mà repo Safe-GS đang ghim. `SAM2_GIT_REF` là commit SAM 2 mà notebook SAM 2 của bạn đã dùng (ghi trong `run_info.json`).

#@markdown ### Chạy
VERBOSE   = False  #@param {type:"boolean"}
OVERWRITE = False  #@param {type:"boolean"}
#@markdown `VERBOSE`: thêm `--verbose` (vẽ hình trung gian, chậm hơn). `OVERWRITE`: xoá `data/<scene>/masks` và `discard` cũ nếu đã có (mặc định báo lỗi để khỏi mất kết quả).

#@markdown ### Tải kết quả về máy
DOWNLOAD_WHEN = "after_each_scene"  #@param ["after_each_scene", "at_end", "never"]
ZIP_CONTENT   = "light"  #@param ["light", "full"]
ZIP_PARTS     = 1  #@param {type:"integer"}
#@markdown `light`: mỗi vật chỉ gồm `masks/*.png` + `points3d.ply` (nhẹ). `full`: cả thư mục `masks/`, `discard/` như sau bước 4 (mỗi vật có bản sao toàn bộ ảnh nên **rất nặng**).

## 2. Môi trường: CUDA 12.x, Python 3.12, thư viện ghim theo `environment.yml`

- **Vì sao dựng venv Python 3.12 riêng:** torch 2.5.1 (CUDA 12.4) chỉ có wheel cho Python ≤ 3.12, còn Colab mới dùng Python 3.13. Venv này **không đụng** môi trường của Colab.
- Extension CUDA `point_projection` (README mục 5) cần `nvcc` 12.x; nếu Colab chưa có sẽ cài từ kho NVIDIA.

In [ ]:
import os, sys, re, glob, json, shutil, subprocess, time, zipfile
from pathlib import Path

def sh(cmd, cwd=None, env=None, check=True, quiet=False):
    """Chạy lệnh bash và in log ngay khi có (không đợi lệnh xong). Lỗi thì nêu 25 dòng cuối."""
    print("$", cmd if len(cmd) < 400 else cmd[:400] + " …")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if not quiet:
            print(line, end="")
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"Lệnh lỗi (mã {rc}). 25 dòng cuối:\n" + "".join(tail[-25:]))
    return rc

def _nvcc_version(home):
    try:
        out = subprocess.run([f"{home}/bin/nvcc", "--version"], capture_output=True, text=True).stdout
        m = re.search(r"release (\d+)\.(\d+)", out)
        return (int(m[1]), int(m[2])) if m else None
    except Exception:
        return None

def find_or_install_cuda12():
    """torch 2.5.1 (CUDA 12.4) chỉ build được extension CUDA nếu nvcc cùng họ 12.x (Colab mới có thể chỉ có nvcc 13)."""
    for home in sorted(glob.glob("/usr/local/cuda-12.*")) + ["/usr/local/cuda"]:
        v = _nvcc_version(home)
        if v and v[0] == 12:
            print("Dùng CUDA", v, "tại", home)
            return home
    osr = open("/etc/os-release").read()
    m = re.search(r'VERSION_ID="(\d+)\.(\d+)"', osr)
    distro = f"ubuntu{m[1]}{m[2]}"
    print(f"Chưa có nvcc 12.x → cài CUDA toolkit 12.x từ kho NVIDIA ({distro}), mất vài phút …")
    sh(f"wget -q https://developer.download.nvidia.com/compute/cuda/repos/{distro}/x86_64/cuda-keyring_1.1-1_all.deb -O /tmp/cuda-keyring.deb"
       f" && dpkg -i /tmp/cuda-keyring.deb && apt-get update -qq")
    for v in ("12-4", "12-6", "12-8"):     # ưu tiên đúng 12.4 như môi trường tác giả; bản mới hơn chỉ khác phiên bản phụ
        if sh(f"apt-get install -y -qq cuda-toolkit-{v}", check=False) == 0:
            home = f"/usr/local/cuda-{v.replace('-', '.')}"
            print("Đã cài", home)
            return home
    raise RuntimeError("Không cài được CUDA toolkit 12.x bằng apt.")

CUDA_HOME = find_or_install_cuda12()
COMPUTE_CAP = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip().splitlines()[0]
print("CUDA_HOME =", CUDA_HOME, "| compute capability", COMPUTE_CAP)

In [ ]:
def make_env(venv, extra_pythonpath=""):
    """Biến môi trường để chạy script trong venv với extension CUDA (README bước 6: LD_LIBRARY_PATH của torch)."""
    env = os.environ.copy()
    env["CUDA_HOME"] = CUDA_HOME
    env["PATH"] = f"{CUDA_HOME}/bin:{venv}/bin:" + env["PATH"]
    env["TORCH_CUDA_ARCH_LIST"] = COMPUTE_CAP
    env["MAX_JOBS"] = "4"
    env["PYTHONUNBUFFERED"] = "1"
    try:
        torch_lib = subprocess.check_output([f"{venv}/bin/python", "-c",
                                             "import torch, os; print(os.path.join(os.path.dirname(torch.__file__), 'lib'))"],
                                            text=True).strip()
        env["LD_LIBRARY_PATH"] = f"{torch_lib}:{CUDA_HOME}/lib64:" + env.get("LD_LIBRARY_PATH", "")
    except Exception:
        pass
    if extra_pythonpath:
        env["PYTHONPATH"] = extra_pythonpath + ":" + env.get("PYTHONPATH", "")
    return env

VENV = Path("/content/venv312")
VPY = VENV / "bin" / "python"
sh("apt-get install -y -qq libgl1 libglib2.0-0 libgomp1", check=False)   # opencv / open3d cần
sh("pip -q install uv gdown")
if not VPY.exists():
    sh(f"uv venv --python 3.12 {VENV}")

def vpip(args, env=None):
    sh(f"uv pip install --python {VPY} {args}", env=env)

# phiên bản theo environment.yml của tác giả
vpip("torch==2.5.1 torchvision==0.20.1")
vpip("numpy==2.1.3 pillow==11.1.0 scipy==1.15.2 scikit-learn==1.6.1 matplotlib==3.10.1 opencv-python==4.11.0.86 "
     "open3d==0.19.0 pycolmap==3.11.1 joblib==1.4.2 pandas==2.2.3 hydra-core==1.3.2 iopath==0.1.10 omegaconf==2.3.0 tqdm")
vpip("setuptools wheel ninja")
sh(f"{VPY} -c \"import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPU:', torch.cuda.is_available())\"")

## 3. Mã nguồn tác giả, checkpoint SAM 2, build extension CUDA

In [ ]:
REPO = Path("/content/Split_and_Splat")
if not REPO.exists():
    sh(f"git clone --recursive {REPO_URL} {REPO}")
sh(f"git -C {REPO} checkout -q {REPO_REF} && git -C {REPO} submodule update --init --recursive -q")
print("Commit:", subprocess.check_output(["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True).strip())

# checkpoint đúng đường dẫn mà mask_propagation.py đọc: <repo>/checkpoints/sam2.1_hiera_large.pt (README mục 3)
ckpt = REPO / "checkpoints" / "sam2.1_hiera_large.pt"
ckpt.parent.mkdir(exist_ok=True)
if not ckpt.exists():
    sh(f"wget -q -O {ckpt} https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt")
print(ckpt, f"{ckpt.stat().st_size / 1e6:.0f} MB")

ENV = make_env(VENV)
# SAM 2 bản chính thức (README mục 3: `git clone facebookresearch/sam2` rồi `pip install -e .`). Bỏ build `_C` (chỉ dùng cho hậu xử lý video).
vpip(f'"git+https://github.com/facebookresearch/sam2.git@{SAM2_GIT_REF}" --no-build-isolation', env={**ENV, "SAM2_BUILD_CUDA": "0"})

# README mục 5: extension CUDA point_projection
sh(f"uv pip install --python {VPY} -e {REPO}/point_projection --no-build-isolation", env=ENV)
ENV = make_env(VENV)   # tính lại LD_LIBRARY_PATH sau khi torch đã có
sh(f"{VPY} -c \"import point_projection_cuda, pycolmap, open3d; print('point_projection_cuda OK | pycolmap', pycolmap.__version__, '| open3d', open3d.__version__)\"",
   cwd=REPO, env=ENV)

## 4. Lấy dữ liệu đầu vào

- **Mask bước 1:** giải nén vào `output/<scene>_autoseg_mask/<tên>/<k>.png` — đúng thư mục mà `mask_propagation.py` đọc (`mask_folder`). Tên thư mục frame bỏ tiền tố `frame_` (`frame_00001` → `00001`), **y như `auto_seg.py` tự làm** khi đổi tên ảnh (`new_name.replace("frame_", "")`).
- **Đầu vào chuẩn bị:** giải nén vào `data/<scene>/` (images, sparse/0, depth).

In [ ]:
import re

def fetch_zips(ids_str, dest):
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    for fid in [x.strip() for x in ids_str.replace("\n", ",").split(",") if x.strip()]:
        before = set(dest.glob("*.zip"))
        sh(f"gdown {fid} -O {dest}/")
        if set(dest.glob("*.zip")) == before:
            print("⚠️ không thấy file zip mới sau khi tải", fid)
    return sorted(dest.glob("*.zip"))

stage1_zips = fetch_zips(STAGE1_GDRIVE_IDS, STAGE1_ZIP_DIR) if STAGE1_GDRIVE_IDS.strip() else sorted(Path(STAGE1_ZIP_DIR).glob("*.zip"))
prep_zips   = fetch_zips(PREP_GDRIVE_IDS, PREP_ZIP_DIR)     if PREP_GDRIVE_IDS.strip()   else sorted(Path(PREP_ZIP_DIR).glob("*.zip"))
print("Zip mask Stage 1 :", [z.name for z in stage1_zips])
print("Zip đầu vào chuẩn bị:", [z.name for z in prep_zips])
assert stage1_zips, "Chưa có zip mask Stage 1 (điền STAGE1_GDRIVE_IDS hoặc upload vào STAGE1_ZIP_DIR)."

In [ ]:
MASK_RE = re.compile(r"^([^/]+)/03_final/masks/([^/]+)/(\d+\.png)$")

def strip_frame_prefix(stem):
    """Quy tắc của sam2/auto_seg.py khi đổi tên ảnh: new_name.replace("frame_", "")."""
    return stem.replace("frame_", "")

def install_stage1_masks(zp, repo):
    """Giải nén mask của 1 zip Stage 1 vào <repo>/output/<scene>_autoseg_mask/<tên>/<k>.png. Trả về (scene, số frame, số mask)."""
    scene, frames, n = None, set(), 0
    with zipfile.ZipFile(zp) as zf:
        for name in zf.namelist():
            m = MASK_RE.match(name)
            if not m:
                continue
            scene = m[1]
            stem = strip_frame_prefix(m[2])
            out = Path(repo) / "output" / f"{scene}_autoseg_mask" / stem / m[3]
            out.parent.mkdir(parents=True, exist_ok=True)
            out.write_bytes(zf.read(name))
            frames.add(stem); n += 1
        # frame không có mask nào vẫn phải có thư mục (code gốc os.listdir thư mục đó)
        for name in zf.namelist():
            m2 = re.match(r"^([^/]+)/03_final/meta/([^/]+)\.json$", name)
            if m2:
                scene = m2[1]
                (Path(repo) / "output" / f"{scene}_autoseg_mask" / strip_frame_prefix(m2[2])).mkdir(parents=True, exist_ok=True)
                frames.add(strip_frame_prefix(m2[2]))
    return scene, len(frames), n

STAGE1 = {}
for zp in stage1_zips:
    scene, nf, nm = install_stage1_masks(zp, REPO)
    if scene:
        STAGE1[scene] = {"frames": nf, "masks": nm}
        print(f"{zp.name}: scene '{scene}' — {nf} frame, {nm} mask")

# đầu vào chuẩn bị → <repo>/data/<scene>/
PREP = {}
for zp in prep_zips:
    with zipfile.ZipFile(zp) as zf:
        tops = sorted({n.split("/")[0] for n in zf.namelist() if n.strip("/")})
        for top in tops:
            if (REPO / "data" / top / "sparse" / "0").exists():
                print(f"{zp.name}: data/{top} đã có → bỏ qua giải nén")
            else:
                members = [n for n in zf.namelist() if n.split("/")[0] == top]
                zf.extractall(REPO / "data", members)
                print(f"{zp.name}: giải nén data/{top} ({len(members)} mục)")
            PREP[top] = zp.name

want = None if SCENES.strip().lower() == "all" else {s.strip() for s in SCENES.split(",") if s.strip()}
common = sorted(set(STAGE1) & set(PREP))
if want is not None:
    common = [s for s in common if s in want]
print("\nScene có đủ cả mask Stage 1 và đầu vào chuẩn bị:", common)
missing = sorted(set(STAGE1) - set(PREP))
assert common, ("Chưa có scene nào đủ đầu vào. Còn thiếu đầu vào chuẩn bị (COLMAP + depth + point cloud) cho: "
                f"{missing}. Chạy notebook chuẩn bị trước rồi điền PREP_GDRIVE_IDS / upload vào {PREP_ZIP_DIR}.")
SCENE_LIST = common

## 5. Kiểm tra đầu vào từng scene

`mask_propagation.py` ghép ảnh, depth, mask và camera COLMAP theo **tên** và **kích thước**; sai một chỗ là lỗi (hoặc tệ hơn: lệch âm thầm).
Ô này kiểm tra trước khi chạy: số lượng khớp, camera là `PINHOLE`, và kích thước ảnh = camera COLMAP = depth = mask.

In [ ]:
CHECK_PY = r"""
import sys, json, os, re
import numpy as np
from PIL import Image
import pycolmap
scene = sys.argv[1]
root = f"data/{scene}"
mask_root = f"output/{scene}_autoseg_mask"
rec = pycolmap.Reconstruction(f"{root}/sparse/0")
res = {"scene": scene, "colmap_images": len(rec.images), "colmap_points3D": len(rec.points3D),
       "camera_models": sorted({c.model.name for c in rec.cameras.values()}), "problems": []}
P = res["problems"]
if res["camera_models"] != ["PINHOLE"]:
    P.append(f"camera phải là PINHOLE (propagation đọc params thành ma trận 2x2), đang là {res['camera_models']}")
ply = f"{root}/sparse/0/points3D.ply"
if not os.path.exists(ply):
    P.append("thiếu sparse/0/points3D.ply")
else:
    with open(ply, "rb") as f:
        head = f.read(2000).decode("latin1")
    m = re.search(r"element vertex (\d+)", head)
    res["ply_points"] = int(m[1]) if m else None
names = sorted(img.name for img in rec.images.values())
miss_img, miss_depth, miss_mask, size_bad = [], [], [], []
for n in names:
    stem = n.rsplit(".", 1)[0]
    if not os.path.exists(f"{root}/images/{stem}.JPEG"): miss_img.append(stem)
    if not os.path.exists(f"{root}/depth/{stem}_pred.npy"): miss_depth.append(stem)
    if not os.path.isdir(f"{mask_root}/{stem}"): miss_mask.append(stem)
for key, lst in (("thiếu images/<tên>.JPEG", miss_img), ("thiếu depth/<tên>_pred.npy", miss_depth), ("thiếu thư mục mask", miss_mask)):
    if lst: P.append(f"{key}: {len(lst)} frame, ví dụ {lst[:3]}")
by_name = {img.name: img for img in rec.images.values()}
show = {names[0], names[len(names) // 2], names[-1]}      # chỉ để in ví dụ; KIỂM TRA áp dụng cho mọi frame
shapes, bad_size = [], 0
for n in names:
    stem = n.rsplit(".", 1)[0]
    cam = by_name[n].camera
    W, H = cam.width, cam.height
    try:
        iw, ih = Image.open(f"{root}/images/{stem}.JPEG").size                         # chỉ đọc header
        d = np.load(f"{root}/depth/{stem}_pred.npy", mmap_mode="r")                    # chỉ đọc header
        ms = sorted(os.listdir(f"{mask_root}/{stem}"))
        mw, mh = Image.open(f"{mask_root}/{stem}/{ms[0]}").size if ms else (W, H)
    except Exception as e:
        P.append(f"không đọc được dữ liệu của {stem}: {e}"); continue
    if [iw, ih] != [W, H] or list(d.shape) != [H, W] or [mw, mh] != [W, H]:
        bad_size += 1
        if bad_size <= 5:
            P.append(f"{stem}: kích thước lệch — camera {W}x{H}, ảnh {iw}x{ih}, depth (H,W)={tuple(d.shape)}, mask {mw}x{mh}")
    if n in show:
        d = np.asarray(d)
        shapes.append({"frame": stem, "camera_WxH": [W, H], "image_WxH": [iw, ih], "depth_HxW": list(d.shape), "mask_WxH": [mw, mh],
                       "depth_dtype": str(d.dtype), "depth_min": float(np.nanmin(d)), "depth_max": float(np.nanmax(d))})
if bad_size > 5:
    P.append(f"… và {bad_size - 5} frame lệch kích thước khác")
res["samples"] = shapes
print(json.dumps(res))
"""
(REPO / "check_inputs_tmp.py").write_text(CHECK_PY, encoding="utf-8")

CHECKS = {}
for scene in SCENE_LIST:
    out = subprocess.run([str(VPY), "check_inputs_tmp.py", scene], cwd=REPO, env=ENV, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(f"[{scene}] kiểm tra đầu vào lỗi:\n{out.stderr[-1500:]}")
    CHECKS[scene] = json.loads(out.stdout.strip().splitlines()[-1])
(REPO / "check_inputs_tmp.py").unlink()

import pandas as pd
rows = [{"scene": s, "ảnh COLMAP": c["colmap_images"], "điểm 3D (ply)": c.get("ply_points"), "camera": ",".join(c["camera_models"]),
         "mask Stage 1 (frame)": STAGE1[s]["frames"], "vấn đề": len(c["problems"])} for s, c in CHECKS.items()]
display(pd.DataFrame(rows).set_index("scene"))
for s, c in CHECKS.items():
    for smp in c["samples"]:
        print(s, smp)
    for p in c["problems"]:
        print("  ❌", s, "—", p)
bad = {s: c["problems"] for s, c in CHECKS.items() if c["problems"]}
assert not bad, "Đầu vào chưa khớp (xem các dòng ❌ ở trên). Sửa ở notebook chuẩn bị rồi chạy lại ô 4."
print("✅ Đầu vào khớp.")

## 6. Bước 2 — lan truyền mask bằng depth: `mask_propagation.py`

Lệnh gốc: `python ./sam2/mask_propagation.py -dataset <scene>` (chạy ở thư mục gốc repo, đọc `./data`, `./output`).
Kết quả: `output/<scene>_masks/<id>/` — mỗi `<id>` là một vật, chứa mask `<tên>.png` ở từng frame nó xuất hiện và `label_<id>.ply`.

In [ ]:
STEP2 = {}
for scene in SCENE_LIST:
    out_dir = REPO / "output" / f"{scene}_masks"
    done_dir = REPO / "data" / scene / "masks"
    if done_dir.exists() and not OVERWRITE:
        print(f"[{scene}] data/{scene}/masks đã có (bước 2–3 đã xong) → bỏ qua. Đặt OVERWRITE = True để làm lại từ bước 2.")
        continue
    if OVERWRITE:
        shutil.rmtree(out_dir, ignore_errors=True)
    elif out_dir.exists() and any(out_dir.iterdir()):
        print(f"[{scene}] output/{scene}_masks đã có → bỏ qua bước 2 (xoá thư mục này nếu muốn chạy lại)")
        continue
    print(f"\n===== [{scene}] bước 2: mask_propagation.py =====")
    t0 = time.time()
    sh(f"{VPY} ./sam2/mask_propagation.py -dataset {scene}" + (" --verbose" if VERBOSE else ""), cwd=REPO, env=ENV)
    STEP2[scene] = round(time.time() - t0)
    n_ids = sum(1 for p in out_dir.iterdir() if p.is_dir()) if out_dir.exists() else 0
    print(f"[{scene}] xong sau {STEP2[scene]}s → {n_ids} vật (ID) trong output/{scene}_masks")
    assert n_ids > 0, f"[{scene}] bước 2 không tạo ra vật nào."

## 7. Bước 3 — chuyển mask vào thư mục dữ liệu

Lệnh gốc: `mv ./output/<scene>_masks ./data/<scene>/masks`.

In [ ]:
for scene in SCENE_LIST:
    src, dst = REPO / "output" / f"{scene}_masks", REPO / "data" / scene / "masks"
    if dst.exists() and not src.exists():
        print(f"[{scene}] data/{scene}/masks đã có → bỏ qua bước 3")
        continue
    if dst.exists():
        if not OVERWRITE:
            raise RuntimeError(f"[{scene}] {dst} đã tồn tại. Đặt OVERWRITE = True để xoá rồi làm lại, hoặc bỏ qua scene này.")
        shutil.rmtree(dst)
        shutil.rmtree(REPO / "data" / scene / "discard", ignore_errors=True)
    if not src.exists():
        raise RuntimeError(f"[{scene}] không thấy {src}: chạy lại bước 2 trước.")
    sh(f"mv ./output/{scene}_masks ./data/{scene}/masks", cwd=REPO)
    print(f"[{scene}] đã chuyển → data/{scene}/masks ({sum(1 for p in dst.iterdir() if p.is_dir())} thư mục vật)")

## 8. Bước 4 — dựng thư mục huấn luyện từng vật: `prepare_folder.sh`

Lệnh gốc: `./bash_dir_utils/prepare_folder.sh <scene>`. Với mỗi vật nó sao chép `images/` và `sparse/`, gom mask `.png` vào `masks/`, đổi `label_<id>.ply` thành `points3d.ply`,
và chuyển vật nào có **dưới 2 mask** sang `data/<scene>/discard/`.

In [ ]:
for scene in SCENE_LIST:
    base = REPO / "data" / scene
    already = any((d / "points3d.ply").exists() for d in (base / "masks").iterdir() if d.is_dir()) or (base / "discard").exists()
    if already:
        # chạy prepare_folder.sh lần 2 sẽ `cp -r images` LỒNG vào thư mục vật (images/images) → không chạy lại
        print(f"[{scene}] đã chạy prepare_folder.sh trước đó → bỏ qua (OVERWRITE = True ở bước 3 để làm lại từ đầu)")
    else:
        print(f"\n===== [{scene}] bước 4: prepare_folder.sh =====")
        sh(f"chmod +x ./bash_dir_utils/prepare_folder.sh && ./bash_dir_utils/prepare_folder.sh {scene}", cwd=REPO, quiet=True)
    kept = sorted((p for p in (base / "masks").iterdir() if p.is_dir()), key=lambda p: int(p.name) if p.name.isdigit() else 10**9)
    disc = sorted(p for p in (base / "discard").iterdir() if p.is_dir()) if (base / "discard").exists() else []
    print(f"[{scene}] giữ {len(kept)} vật, loại {len(disc)} vật (dưới 2 mask)")

## 9. Kết quả từng scene

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def id_color(i):
    rng = np.random.default_rng(int(i) * 7919 + 12345)
    return np.array(rng.integers(40, 256, size=3), dtype=np.uint8)

SUMMARY = []
for scene in SCENE_LIST:
    base = REPO / "data" / scene
    inst = {}
    for d in sorted((p for p in (base / "masks").iterdir() if p.is_dir()), key=lambda p: int(p.name) if p.name.isdigit() else 10**9):
        pngs = sorted((d / "masks").glob("*.png"))
        inst[d.name] = {"frames": [p.stem for p in pngs], "ply": (d / "points3d.ply").exists(), "dir": d}
    ndisc = sum(1 for p in (base / "discard").iterdir() if p.is_dir()) if (base / "discard").exists() else 0
    nfr = [len(v["frames"]) for v in inst.values()] or [0]
    SUMMARY.append({"scene": scene, "vật giữ": len(inst), "vật loại": ndisc, "frame/vật min": min(nfr),
                    "frame/vật tb": round(float(np.mean(nfr)), 1), "frame/vật max": max(nfr),
                    "vật thiếu points3d.ply": sum(not v["ply"] for v in inst.values())})
    # xem bằng mắt: cùng ID có đúng là cùng một vật ở nhiều góc nhìn không?
    all_frames = sorted({f for v in inst.values() for f in v["frames"]})
    if all_frames:
        pick = [all_frames[i] for i in np.linspace(0, len(all_frames) - 1, min(4, len(all_frames))).astype(int)]
        fig, axes = plt.subplots(1, len(pick), figsize=(5 * len(pick), 4), squeeze=False)
        for ax, fr in zip(axes[0], pick):
            img = np.array(Image.open(base / "images" / f"{fr}.JPEG").convert("RGB")).astype(np.float32)
            out = img * 0.45
            for k, v in inst.items():
                mp = v["dir"] / "masks" / f"{fr}.png"
                if mp.exists():
                    m = np.array(Image.open(mp).convert("L")) > 0
                    out[m] += id_color(k) * 0.55
            ax.imshow(np.clip(out, 0, 255).astype(np.uint8)); ax.set_title(f"{scene} / {fr}"); ax.axis("off")
        plt.tight_layout(); plt.show()
display(pd.DataFrame(SUMMARY).set_index("scene"))

## 10. Nén và tải về máy

In [ ]:
def zip_scene(scene, content="light", parts=1):
    base = REPO / "data" / scene
    parts = max(1, int(parts))
    zdir = Path("/content") if Path("/content").is_dir() else Path(".")
    files = []
    if content == "full":
        for sub in ("masks", "discard"):
            files += [f for f in (base / sub).rglob("*") if f.is_file()] if (base / sub).exists() else []
    else:   # light: mỗi vật chỉ giữ masks/*.png + points3d.ply
        for sub in ("masks", "discard"):
            if not (base / sub).exists():
                continue
            for d in (base / sub).iterdir():
                if d.is_dir():
                    files += [f for f in (d / "masks").glob("*.png")]
                    if (d / "points3d.ply").exists():
                        files.append(d / "points3d.ply")
    insts = sorted({f.relative_to(base).parts[:2] for f in files})
    idx_of = {k: i * parts // max(1, len(insts)) for i, k in enumerate(insts)}
    zips = [zdir / (f"{scene}_split_instances_{content}" + (f"_part{k + 1}of{parts}" if parts > 1 else "") + ".zip") for k in range(parts)]
    hs = [zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED, compresslevel=1) for z in zips]
    try:
        for f in sorted(files):
            rel = f.relative_to(base)
            hs[idx_of.get(rel.parts[:2], 0)].write(f, arcname=str(Path(scene) / rel))
    finally:
        for h in hs:
            h.close()
    for z in zips:
        mb = z.stat().st_size / 1e6
        print(f"  {z.name}: {mb:.0f} MB" + ("  ⚠️ lớn, nên tăng ZIP_PARTS" if mb > 1500 else ""))
    return zips

def download_scene(scene):
    zips = zip_scene(scene, ZIP_CONTENT, ZIP_PARTS)
    from google.colab import files
    for z in zips:
        files.download(str(z))

if DOWNLOAD_WHEN != "never":
    for scene in SCENE_LIST:
        print(scene)
        download_scene(scene)
else:
    print("DOWNLOAD_WHEN = never")

## Sau notebook này

`data/<scene>/masks/<id>/` đã sẵn sàng cho **Stage 2 — Instance Reconstruction** của Split&Splat (`run_all.sh`, `run_ref.sh`, …): train 3DGS riêng cho từng vật.
Zip `light` chỉ chứa mask + `points3d.ply` từng vật; ảnh và `sparse/` lấy lại từ `data/<scene>/` (chính `prepare_folder.sh` cũng chỉ sao chép chúng vào từng thư mục vật).